# 02. A1 の構築とトレース取得

`build_agent()` で A1 を組み立て、`TracingRunner` で実行トレースを構造化する。

`A1.go_stream()` は使わない。あれは `pretty_print()` を通した整形済み文字列しか返さず、
根拠抽出に必要な情報（生コード・生 observation・メッセージ種別）が失われるため
（docs/design/02 §2.3）。ここでは `agent.app.stream()` を直接回す。

Ollama と biomni が無い場合は、末尾のフォールバックセルで
同じコードパスをフェイクエージェントで通せる。

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
from biomni_hypo.config import Settings, apply_biomni_env
from biomni_hypo.llm import ollama_status
from biomni_hypo.policy import ResourcePolicy

settings = Settings()
apply_biomni_env(settings)
policy = ResourcePolicy.load(settings.policy_path)

READY = ollama_status(settings.ollama_base_url).reachable
print("Ollama:", READY)

## A1 を組み立てる

`build_agent()` の中で以下がすべて適用される（1 箇所に閉じてある）:

| 対策 | 内容 |
| --- | --- |
| §4.1/4.2 | `agent.llm` を stop・num_ctx・base_url 付きの `ChatOllama` に差し替え |
| §4.4 | `expected_data_lake_files` を明示してデータレイクの一括ダウンロードを抑止 |
| §4.5 | `module2api` を仮説構築で使うモジュールだけに絞る |
| §5.2 | 拒否ツールを `tool_registry` と `module2api` の両方から削除 |

In [ ]:
from biomni_hypo.agent_factory import build_agent

bundle = build_agent(settings, policy)   # 初回は数十秒かかる
print(bundle.report)

In [ ]:
# 拒否ツールが本当に消えているか確認する
denied = policy.denied_tool_names()
remaining = {a["name"] for apis in bundle.agent.module2api.values() for a in apis}
print("拒否ツール:", denied)
print("残存      :", sorted(set(denied) & remaining) or "なし ✅")
print("システムプロンプトに出現:", [t for t in denied if t in bundle.agent.system_prompt] or "なし ✅")

## トレースを取る

`max_steps` を小さくして、まずは短いタスクで形を確認する。

In [ ]:
from biomni_hypo.tracing import TracingRunner

settings.max_steps = 12
QUESTION = "ヒトの TP53 遺伝子の Ensembl 上の位置と、既知の主要な機能を調べてください。"

runner = TracingRunner(bundle, run_id="nb02")
for step in runner.iter_steps(QUESTION):
    head = (step.code or step.text).strip().replace("\n", " ")[:90]
    tools = ",".join(t.name for t in step.tools)
    print(f"[{step.idx:2d}] {step.kind.value:15s} {head}")
    if tools:
        print(f"      tools: {tools}")
    if step.datasets:
        print(f"      data : {step.datasets}")
    if step.citations:
        print(f"      cite : {[c.identifier for c in step.citations]}")

In [ ]:
trace = runner.result()
print("ステップ数           :", len(trace.steps))
print("打ち切り理由         :", trace.stopped_reason or "なし")
print("observation 自己生成 :", trace.hallucinated_observations, "← 0 でなければ 01 に戻る")
print("結論                 :", trace.solution_text[:200])

## 3 段階の「使用データ」（docs/design/03 §3.2）

- **A. 検討対象**: リソース検索が選んだもの（`use_tool_retriever=True` のときのみ）
- **B. 実際に触れた**: コードが読み込み・呼び出ししたもの
- **C. 主張を支えた**: 03 の抽出フェーズで仮説に紐付いたもの

C ⊆ B ⊆ A が崩れたら幻覚のシグナル。

In [ ]:
from biomni_hypo.pipeline import collect_resources

print("A. 検討対象:", trace.resources_considered or "(リソース検索 OFF)")
print()
print("B. 実際に触れた:")
for r in collect_resources(trace.steps, policy):
    mark = "⚠️" if r.review_required else "✅"
    print(f"  {mark} [{r.kind.value:9s}] {r.name:44s} {r.license:14s} steps={r.step_idxs}")

## フォールバック: Ollama / biomni なしでコードパスだけ確認する

CI やオフライン環境用。`TracingRunner` の分類ロジックは同じものが走る。

In [ ]:
from biomni_hypo.fixtures import TRACE_MESSAGES, FakeA1Module, fake_bundle

fake = fake_bundle(TRACE_MESSAGES, settings=settings, policy=policy)
fake_trace = TracingRunner(fake, run_id="nb02_fake", guard_module=FakeA1Module()).run("フェイク質問")

for s in fake_trace.steps:
    print(f"[{s.idx}] {s.kind.value:12s} tools={[t.name for t in s.tools]} data={s.datasets} cite={len(s.citations)}")